In [ ]:
!pip install -qU "langchain[google-genai]"
!pip install -qU langchain_community langchain-pymupdf4llm
!pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph
!pip install -U langchain-huggingface
!pip install supabase
!pip install -qU langchain-google-genai
!pip install -qU langchain-core
!pip install -qU langchain_postgres

In [ ]:
import getpass
import os

os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)

from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [ ]:
# %%
# rag_agent_app.ipynb
# Cell 1: Imports & Environment Setup

import os
import traceback

# --- Dependencies ---
from sentence_transformers.cross_encoder import CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# --- Environment Variable & Setup ---
# IMPORTANT: It's recommended to set this as a true environment variable.
if "GOOGLE_API_KEY" not in os.environ:
    # --- ⬇️ PASTE YOUR GOOGLE API KEY HERE ⬇️ ---
    os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
    # -------------------------------------------
    print("WARNING: Using a hardcoded GOOGLE_API_KEY. It's better to set this as a system environment variable.")

In [ ]:
# %%
# Cell 2: Configuration

CONNECTION_STRING = (__import__('sys').path.insert(0, '..') or __import__('config').DB_CONNECTION_STRING)
COLLECTION_NAME = "VectorDB"

In [ ]:
# %%
# Cell 3: RAG Agent Class Definition (Prompt Agnostic + Fallback Handling)

import os
import traceback

# --- Dependencies ---
from sentence_transformers.cross_encoder import CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from sqlalchemy import create_engine  # <-- Required for pooled connection

class RAGAgent:
    """
    A ReAct-based RAG agent that connects to a pre-existing vector store
    to answer complex, multi-hop questions. It supports persistent, stateful
    chat history.
    """
    def __init__(self):
        print("--- Initializing RAG Agent ---")
        self._initialize_models()
        self.engine = self._create_db_engine()
        self._setup_retriever(self.engine)
        self.agent_tool = self._create_agent_tool()
        self.checkpointer = MemorySaver()
        self._compile_graph()
        print("--- RAG Agent Initialized Successfully ---")

    def _initialize_models(self):
        print("  -> Loading models...")
        self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
        print("    -> Gemini model loaded.")
        self.cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        print("    -> Cross-Encoder model loaded.")

    def _create_db_engine(self):
        print("  -> Creating database engine with connection pooling...")
        engine = create_engine(CONNECTION_STRING, pool_size=5, max_overflow=2, pool_recycle=300)
        print("    -> Database engine created.")
        return engine

    def _setup_retriever(self, engine):
        print("  -> Setting up retriever...")
        embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
        try:
            vector_store = PGVector(
                embeddings=embeddings,
                collection_name=COLLECTION_NAME,
                connection=engine,
            )
            print("    -> Successfully connected to the existing PGVector store.")
            self.retriever = vector_store.as_retriever(search_kwargs={"k": 10})
            print("  -> Retriever created successfully.")
        except Exception as e:
            print(f"\nERROR: Could not connect to the PGVector store: {e}")
            raise

    def _create_agent_tool(self):
        """Creates the search-and-rerank tool for the ReAct agent."""
        @tool
        def retrieve_and_rerank_context(query: str) -> str:
            """
            Retrieves and re-ranks the most relevant passages from the FTP 2023 document.
            If no documents are relevant, returns a fallback message instructing
            the LLM to infer based on policy goals and structure.
            """
            print(f"---EXECUTING AGENT TOOL with query: '{query}'---")
            retrieved_docs = self.retriever.invoke(query)

            if not retrieved_docs:
                return (
                    "No directly matching policy content was found in the FTP 2023 documents. "
                    "However, based on general export policy goals, consider suggesting applicable infrastructure, "
                    "logistics, or district-level support mechanisms that align with the user's query."
                )

            pairs = [[query, doc.page_content] for doc in retrieved_docs]
            scores = self.cross_encoder.predict(pairs)
            scored_docs = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)
            top_n = 3
            re_ranked_docs = [doc for _, doc in scored_docs[:top_n]]
            print(f"  -> Selected top {len(re_ranked_docs)} documents after re-ranking.")
            context_parts = []
            for i, doc in enumerate(re_ranked_docs):
                context_parts.append(
                    f"--- Passage {i+1} ---\n"
                    f"Source: {doc.metadata.get('source_file', 'N/A')}\n"
                    f"Section: {doc.metadata.get('section_header', 'N/A')}\n"
                    f"Content: {doc.page_content}"
                )
            return "\n\n".join(context_parts)
        return retrieve_and_rerank_context

    def _compile_graph(self):
        print("  -> Compiling ReAct agent graph...")
        self.graph = create_react_agent(
            self.llm,
            [self.agent_tool],
            checkpointer=self.checkpointer
        )
        print("  -> Graph compiled.")

    def _augment_prompt(self, user_input: str) -> str:
        """
        Augments the prompt dynamically with reasoning instructions without hardcoding it into the agent.
        """
        reasoning_prefix = (
            "You are a policy advisor assisting with implementation of India's Foreign Trade Policy (FTP) 2023. "
            "If a user's query includes terms not explicitly mentioned in the document (e.g., 'cold chain', "
            "'Niryat Bandhu'), infer an answer based on related concepts like export infrastructure, DEAP, DEPC roles, "
            "district support mechanisms, and the general objectives of the FTP.\n\n"
        )
        return f"{reasoning_prefix}User question: {user_input}"

    def invoke(self, question: str, thread_id: str, augment: bool = True):
        """
        Runs the agent for a given question and conversation thread.

        Args:
            question: User's raw question.
            thread_id: Unique ID for the session.
            augment: Whether to prepend reasoning instructions to the input dynamically.
        """
        config = {"configurable": {"thread_id": thread_id}}
        print(f"\n--- Running Agent for thread '{thread_id}' ---")
        final_answer = ""

        formatted_prompt = self._augment_prompt(question) if augment else question

        for event in self.graph.stream({"messages": [HumanMessage(content=formatted_prompt)]}, config=config, stream_mode="values"):
            last_message = event["messages"][-1]
            print("\n" + "="*50)
            if isinstance(last_message, AIMessage):
                print("LLM Response:")
            else:
                print("Tool Output:")
            last_message.pretty_print()
            if isinstance(last_message, AIMessage) and not last_message.tool_calls:
                final_answer = last_message.content

        print("\n\n" + "="*60)
        print("--- FINAL RESPONSE ---")
        print(f"Thread ID: {thread_id}")
        print(f"Question: {question}")
        print(f"Answer:\n{final_answer}")
        print("="*60 + "\n")
        return final_answer


In [ ]:
thread_id = "trade_convo_final_v_005"
agent = RAGAgent()
agent.invoke(question="Voluntary Self Disclosure of export of", thread_id=thread_id)

In [ ]:
agent.invoke(question="I’m drafting a District Export Action Plan for a coastal district with high marine export potential. What kind of infrastructure and policy interventions should I include to support cold chain logistics from fishery to port?", thread_id=thread_id)